# PACCE — Examples

This is an updated and upgraded version of the PACCE (Perl algorithm to compute continuum and equivalent widths, Riffel & Vale, 2011, Ap&SS.334..351) written in python.

Here we present a series of examples of different utilities you may find useful when measuring stellar population indices as well as comparing models and observations.
PACCE is a python code based upon standard python libraries such as numpy, scipy and (to use the wrapper) pandas.

All examples below use the SDSS data bundled in `pacce/examples/`.

In [ ]:
from pacce import pacce
import os

# Resolve paths to the bundled example data
import pacce as pacce_pkg
examples_dir = os.path.join(os.path.dirname(pacce_pkg.__file__), 'examples')
suport_dir   = os.path.join(os.path.dirname(pacce_pkg.__file__), 'suport_files')
print('Examples dir:', examples_dir)
print('Support dir:', suport_dir)

## 1. Basic index measurement

We implemented a wrapper to the code in order to improve the user experience by letting the code handle the different steps involved in the spectra preparations and measurements.
You need the following:
- A table containing the name of spectra you want to measure. This can be as simple as a direct list of all the file names or a more complex table containing relevant information (e.g. redshifts, velocity dispersions).
- A directory containing the files you mention in the table.
- An index definition file where you mention all the indices you would like to measure in your spectra. You can find example files in the `suport_files` directory. Each definition needs 4 things: a name, a wavelength range for the feature, continuum bands, and a reference. Example:

`name |     definition      |         cont1,               cont2,...     |  reference`

`Mgb |  5160.1250-5192.6250 |  5142.6250-5161.3750, 5191.3750-5206.3750  |  (Trager+98)`
- An output file to write the table containing the measurements.

An optional keyword that makes the code cleaner is the `print_log` option, where you can redirect the prints from the program.

The code returns a `pandas.DataFrame` containing all the relevant info. Let's give it a try:

In [ ]:
a = pacce(
    filename=os.path.join(examples_dir, 'sdss_table_example.dat'),
    path_to_files=os.path.join(examples_dir, 'sdss_example'),
    IndexDefs=os.path.join(suport_dir, 'less_defs.ind'),
    output_file='measurements_basic.txt',
    print_log='basic_log.txt'
)
a.head()

## 2. Spectral convolution

The code can do much more! In this example we add two additional keywords:
-  `sigma_fin` is the final velocity dispersion (km/s) that you want the spectra broadened to.

If you provide both an initial and a final resolution (`_ini` and `_fin`), the code will perform the convolution of the spectra before measuring the indices.

A nice feature of the code is that you can add this info as a column in the input table, i.e. a column named `sigma` will be interpreted as the `sigma_ini` of that spectrum. This is useful if you have data with different resolutions and want them all measured at the same resolution — just provide a `_fin` value and done!

Notice in the SDSS example table there is a column named `sigma`, so the code takes this info as the `_ini` value:

In [ ]:
b = pacce(
    filename=os.path.join(examples_dir, 'sdss_table_example.dat'),
    path_to_files=os.path.join(examples_dir, 'sdss_example'),
    IndexDefs=os.path.join(suport_dir, 'less_defs.ind'),
    output_file='measurements_convolved.txt',
    sigma_fin=300,
    print_log='convolution_log.txt'
)
b.head()

## 3. Magnitude conversion and composite indices

You can also convert indices from Å to magnitudes with `A_to_mag`, and add new composite indices that are combinations of other indices (e.g. MgFe') with `compute_idx`. The `compute_idx` file should contain one expression per line using the pandas `.eval()` syntax.

In [ ]:
c = pacce(
    filename=os.path.join(examples_dir, 'sdss_table_example.dat'),
    path_to_files=os.path.join(examples_dir, 'sdss_example'),
    IndexDefs=os.path.join(suport_dir, 'less_defs.ind'),
    output_file='measurements_mag.txt',
    sigma_fin=300,
    A_to_mag=['Mg2', 'Mg1'],
    compute_idx=os.path.join(examples_dir, 'compute.txt'),
    print_log='mag_log.txt'
)
c.head()

## 4. Error estimation

PACCE can deal with uncertainties in the spectra. If you provide an error spectrum (third column in the data file), the code will provide errors for the measurements. This can be done in two ways:

1. **Analytical (default)**: The code estimates the error by calculating the S/N in the absorption feature and using the equations in [Cardiel et al. (2006)](https://arxiv.org/pdf/astro-ph/0606341.pdf). This is faster but does not account for continuum variation.

2. **Monte Carlo (`simulate=N`)**: The code generates `N` mock spectra from the observed spectrum using the error spectrum, re-measures the indices, and uses the standard deviation as the error. More robust but slower.

In [ ]:
# Monte Carlo error estimation with 50 iterations
d = pacce(
    filename=os.path.join(examples_dir, 'sdss_table_example.dat'),
    path_to_files=os.path.join(examples_dir, 'sdss_example'),
    IndexDefs=os.path.join(suport_dir, 'less_defs.ind'),
    output_file='measurements_mc.txt',
    sigma_fin=300,
    simulate=50,
    compute_idx=os.path.join(examples_dir, 'compute.txt'),
    print_log='mc_log.txt'
)
d.head()

## 5. Diagnostic plots

You can visualise the continuum fit and band locations for each index. Just provide a directory with `path_singleind_plots` and the code will save a plot per index per spectrum. Be mindful that this takes significantly longer.

You can also plot all indices for each spectrum in a single figure using `AllIndicesPlot` and `allindices_plot_path`.

In [ ]:
e = pacce(
    filename=os.path.join(examples_dir, 'sdss_table_example.dat'),
    path_to_files=os.path.join(examples_dir, 'sdss_example'),
    IndexDefs=os.path.join(suport_dir, 'less_defs.ind'),
    output_file='measurements_plots.txt',
    sigma_fin=300,
    compute_idx=os.path.join(examples_dir, 'compute.txt'),
    print_log='plots_log.txt',
    path_singleind_plots='./pacce_figures/',
    AllIndicesPlot='AllIndicesPlot.png',
    allindices_plot_path='./all_indices/'
)
e.head()